## Installing dependencies and setup

In [4]:
#------------------------------------------------
# INSTALLING DEPENDENCIES AND STARTING OLLAMA SERVER
#------------------------------------------------

!nvidia-smi
!apt-get update && apt-get install -y pciutils
!sudo apt-get install zstd 
!pip install deepeval rich pandas tabulate --quiet
!curl -fsSL https://ollama.com/install.sh | sh
!pkill ollama # just to make sure there isn't any existing server running
!pip install ollama

import pathlib
import subprocess
import time
import json
import re
import os
import ollama
from ollama import Client
import httpx
from pydantic import BaseModel
from google.colab import drive

server_process = subprocess.Popen(["ollama", "serve"])

Thu May  7 18:30:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!git clone https://www.github.com/rryyyymmmmmmmm/pfe_llm_eval
!mv pfe_llm_eval/* . && rm -rf pfe_llm_eval

#-----------------------------------------------------------------
# LOADING THE SYSTEM PROMPT
#-----------------------------------------------------------------

prompt = pathlib.Path("system_prompt.txt").read_text(encoding="utf-8")

#-----------------------------------------------------------------
# LOADING THE OCR RESULTS TO BE USED IN THE EXAMPLES AND EVALUATION
#-----------------------------------------------------------------

ordos_ocr = {}

for x in pathlib.Path("ocr_output").glob("*.txt"):
    ordos_ocr[str(x).removeprefix("ocr_output/").removesuffix("_recto_preprocessed.jpg_ocr.txt")] = x.read_text(encoding="utf-8")


# ─────────────────────────────────────────────────────────────
# SYSTEM PROMPT and USER TEMPLATE
# ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
{prompt}
"""
USER_TEMPLATE = """Extract the structured data from this OCR prescription text:
---
{ocr_text}
---"""

Cloning into 'pfe_llm_eval'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 85 (delta 4), reused 11 (delta 4), pack-reused 73 (from 2)
Receiving objects: 100% (85/85), 105.87 MiB | 31.76 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [6]:
#-----------------------------------------------------------------
# LOADING THE EXAMPLES OF FRONT ORDOS TO BE USED IN THE EVALUATION
#-----------------------------------------------------------------

EXAMPLES = []
#!mkdir EXAMPLES

for ex in pathlib.Path("EXAMPLES").glob("*.json"):
    with open(ex, "r", encoding="utf-8") as f:
        data = json.load(f)
        data["ocr_text"] = ordos_ocr[data.get("id")]
        EXAMPLES.append({
            "name": ex.name.removesuffix("_example.json"),
            "data": data
        })

print(EXAMPLES[0].keys())

KeyError: None

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434"  
OLLAMA_TIMEOUT  = 300  

#─────────────────────────────────────────────────────────────
# PULLING MODELS
#-─────────────────────────────────────────────────────────────

# from the LLMStructBench paper 
LLM_STRUCT_BENCH_MODELS_BIG = [
    "gemma3:12b",

    "deepseek-r1:14b",
    "deepseek-r1:8b",
    "deepseek-r1:7b",
    
    "phi4:14b",
    "phi3:14b",

    "qwen3:14b",
    "qwen3:8b",
    
]

LLM_STRUCT_BENCH_MODELS_SMALL = [
    "qwen3:4b",
    "qwen3:1.7b",
    "qwen3:0.6b",

    "phi4-mini:3.8b",
    "phi3.5:3.8b",
    "phi3:3.8b",

    "deepseek-r1:1.5b",    
    "gemma3:1b",
]


STRUCT_EVAL_MODELS = [
    "llama3.1:8b-instruct-q4_K_M",
    "llama3:8b-instruct-q4_K_M",
    "phi3:3.8b-mini-128k-instruct-q4_K_M",
    "phi4-mini:3.8b-q4_K_M",
    "qwen2.5:7b-instruct-q4_K_M",
    "qwen3:4b-instruct"
]

#--------------------------------------------------- 
# ASSIGNING WHICH MODELS TO EVAL
#---------------------------------------------------

MODELS_TO_EVAL = LLM_STRUCT_BENCH_MODELS_SMALL

for m in MODELS_TO_EVAL:
    os.system(f"ollama pull {m}")

!ollama list # making sure the models were pulled successfully

client = Client(host=OLLAMA_BASE_URL, timeout=OLLAMA_TIMEOUT) # starting the client to interact with the server

## Evaluation metrics

In [ ]:
# ─────────────────────────────────────────────────────────────
# NOTE: Old fuzzy-match metrics (EntityAccuracyMetric,
# MedicationRecallMetric) have been replaced by GEval-powered
# semantic metrics defined in the cell below.
# This cell is intentionally left as a placeholder.
# ─────────────────────────────────────────────────────────────


In [ ]:
import re
import json
from deepeval.metrics import BaseMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# ─────────────────────────────────────────────────────────────
# Helper: fuzzy string match (used only by non-AI metrics)
# ─────────────────────────────────────────────────────────────

def normalise(s) -> str:
    """Strip all non-alphanumeric characters and lowercase."""
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def partial_match(pred, gold) -> float:
    """Score 0–1 for fuzzy string similarity."""
    p, g = normalise(pred), normalise(gold)
    if not g:
        return 1.0 if not p else 0.5
    if not p:
        return 0.0
    if p == g:
        return 1.0
    from difflib import SequenceMatcher
    return SequenceMatcher(None, p, g).ratio()


# ─────────────────────────────────────────────────────────────
# METRIC 1 — JSON Validity  (unchanged, no AI needed)
# ─────────────────────────────────────────────────────────────
class JSONValidityMetric(BaseMetric):
    """
    GOAL 1 — Can the model's output be parsed as JSON at all?

    No AI judge needed: this is a pure syntactic check.
    A score of 1.0 means valid JSON; 0.0 means completely broken.
    The format="json" Ollama flag helps, but doesn't guarantee correctness,
    so this guard is still essential.
    """

    name      = "JSON Validity"
    threshold = 1.0

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            json.loads(test_case.actual_output)
            self.score  = 1.0
            self.reason = "Output is valid, parseable JSON."
        except Exception as e:
            self.score  = 0.0
            self.reason = f"JSON parse failure: {e}"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


# ─────────────────────────────────────────────────────────────
# METRIC 2 — Schema Compliance  (unchanged, no AI needed)
# ─────────────────────────────────────────────────────────────
REQUIRED_KEYS              = {"medical_facility", "patient", "prescription_date", "doctor", "medications"}
MEDICALFACILITY_STRUCT_KEYS = {"name", "type"}
PATIENT_STRUCT_KEYS         = {"full_name", "first_name", "family_name", "date_of_birth", "age"}
DOCTOR_STRUCT_KEYS          = {"name", "specialty", "license_number"}
MED_STRUCT_KEYS             = {"name", "dosage", "form", "posologie", "QSP", "total_quantity", "package_quantity"}


class SchemaComplianceMetric(BaseMetric):
    """
    GOAL 2 — Are all required fields present in the JSON object?

    No AI judge needed: this is a deterministic structural check.
    Validates top-level keys, nested raw_text/structured sub-objects,
    and per-medication field presence.
    Score = fraction of schema checks passed (threshold 0.8).
    """

    name      = "Schema Compliance"
    threshold = 0.8

    def _check_nested(self, data: dict, field: str, structured_keys: set) -> bool:
        obj = data.get(field)
        if not isinstance(obj, dict):
            return False
        structured = obj.get("structured")
        if not isinstance(structured, dict):
            return False
        return structured_keys.issubset(structured.keys())

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            data = json.loads(test_case.actual_output)
        except Exception:
            self.score  = 0.0
            self.reason = "Unparseable JSON — schema check skipped."
            return 0.0

        checks = [
            isinstance(data, dict),
            REQUIRED_KEYS.issubset(data.keys()),
            self._check_nested(data, "medical_facility", MEDICALFACILITY_STRUCT_KEYS),
            self._check_nested(data, "patient",          PATIENT_STRUCT_KEYS),
            self._check_nested(data, "doctor",           DOCTOR_STRUCT_KEYS),
            "prescription_date" in data and (
                data["prescription_date"] is None or
                isinstance(data["prescription_date"], str)
            ),
            isinstance(data.get("medications"), list),
        ]

        meds = data.get("medications", [])
        if meds:
            checks.append(all(
                isinstance(m, dict) and
                isinstance(m.get("structured"), dict) and
                MED_STRUCT_KEYS.issubset(m["structured"].keys())
                for m in meds
            ))

        self.score  = round(sum(checks) / len(checks), 4)
        self.reason = f"{sum(checks)}/{len(checks)} schema checks passed."
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


# ─────────────────────────────────────────────────────────────
# METRIC 3 — Semantic Entity Accuracy  (AI-powered via GEval)
# ─────────────────────────────────────────────────────────────
def build_entity_accuracy_metric() -> GEval:
    """
    GOAL 3 — How accurately do the extracted entities match the ground truth?

    WHY GEval instead of fuzzy matching:
      • OCR text of French prescriptions contains abbreviations ("Mme", "Dr"),
        Latin dosage shorthand ("qd", "bid"), formatting noise, and accent
        variations that simple string distance handles poorly.
      • A doctor's name written as "Dr. Ben Ali" vs "Bénali" should still score
        high — only a semantic judge can decide that.
      • GEval sends both the model's JSON and the ground-truth JSON to an LLM
        judge that scores according to the criteria below, returning a 0–1 score
        with a written rationale.

    Input mapping (what GEval sees):
      • actual_output  → the model's extracted JSON
      • expected_output → the ground-truth JSON

    Evaluation criteria (what the LLM judge looks for):
      1. Patient identity fields (full_name, first_name, family_name, DOB, age)
      2. Prescription date (allow equivalent date formats)
      3. Doctor fields (name, specialty, license_number)
      4. Medical facility fields (name, type)
      5. Per-medication fields (name, dosage, form, posologie, QSP)
    """
    return GEval(
        name="Semantic Entity Accuracy",
        model="gpt-4o",          # LLM judge — swap to your preferred judge model
        threshold=0.7,
        evaluation_params=[
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ],
        criteria=(
            "You are a senior pharmacist and medical-records auditor evaluating "
            "an AI system that extracts structured data from OCR scans of French "
            "medical prescriptions.\n\n"

            "You will receive:\n"
            "  • ACTUAL OUTPUT  — the JSON produced by the AI model.\n"
            "  • EXPECTED OUTPUT — the ground-truth JSON.\n\n"

            "Score the ACTUAL OUTPUT on a scale from 0 to 1 based on how "
            "accurately it captures the following entity groups:\n\n"

            "1. PATIENT IDENTITY (full_name, first_name, family_name, "
            "date_of_birth, age)\n"
            "   — Accept minor OCR noise, accent loss, or name-order swaps "
            "if the identity is unambiguous.\n\n"

            "2. PRESCRIPTION DATE\n"
            "   — Accept equivalent date formats (DD/MM/YYYY vs YYYY-MM-DD etc.).\n\n"

            "3. DOCTOR (name, specialty, license_number)\n"
            "   — Accept French title abbreviations (Dr., Pr.) and minor "
            "spelling variations.\n\n"

            "4. MEDICAL FACILITY (name, type)\n"
            "   — Accept common abbreviations (CHU, Clinique, Cabinet, etc.).\n\n"

            "5. MEDICATIONS — for each drug, evaluate:\n"
            "   • name     : same drug, even if brand vs INN, or minor OCR typo\n"
            "   • dosage   : correct numeric value AND unit (mg, ml, UI, etc.)\n"
            "   • form     : tablet, syrup, injection, etc.\n"
            "   • posologie: dosing schedule — accept Latin shorthands "
            "(qd=once daily, bid=twice, tid=three times)\n"
            "   • QSP      : quantity sufficient — accept equivalent phrasings\n\n"

            "SCORING GUIDE:\n"
            "  1.0 — All entities correct or trivially equivalent.\n"
            "  0.8 — One minor field wrong or missing (e.g., missing specialty).\n"
            "  0.6 — Several fields wrong; core identity and main drug correct.\n"
            "  0.4 — Many fields wrong but JSON structure intact.\n"
            "  0.2 — Almost all fields wrong or hallucinated.\n"
            "  0.0 — Output is empty, unparseable, or completely fabricated.\n\n"

            "Be strict about dosage values and units — a wrong dose in a "
            "medical context is a patient-safety issue. Be lenient about "
            "minor OCR artefacts in names."
        ),
    )


# ─────────────────────────────────────────────────────────────
# METRIC 4 — Semantic Medication Recall  (AI-powered via GEval)
# ─────────────────────────────────────────────────────────────
def build_medication_recall_metric() -> GEval:
    """
    GOAL 4 — Were all medications from the prescription found?

    WHY GEval instead of fuzzy string matching:
      • Drug names in French prescriptions mix INN (generic) and brand names,
        French/Arabic transliterations, and OCR corruption.
        e.g. "AMOXICILLINE 500mg" and "Amoxil" are the same drug;
             "DOLIPRANE" and "Paracétamol" are the same drug.
      • A string-distance metric would wrongly penalise these equivalences.
      • The LLM judge can reason about pharmacological equivalence, which is
        exactly the domain knowledge we need.

    Input mapping:
      • actual_output   → the model's extracted JSON (medications list)
      • expected_output → the ground-truth JSON (medications list)

    The judge only cares about RECALL (were all gold drugs found?),
    NOT precision (we don't penalise extra drugs found beyond ground truth).
    """
    return GEval(
        name="Semantic Medication Recall",
        model="gpt-4o",
        threshold=0.8,
        evaluation_params=[
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ],
        criteria=(
            "You are a clinical pharmacist evaluating an AI extraction system "
            "for French medical prescriptions.\n\n"

            "You will receive:\n"
            "  • ACTUAL OUTPUT  — the model's extracted JSON.\n"
            "  • EXPECTED OUTPUT — the ground-truth JSON.\n\n"

            "YOUR ONLY TASK: determine what fraction of the medications listed "
            "in the EXPECTED OUTPUT were successfully identified in the "
            "ACTUAL OUTPUT.\n\n"

            "MATCHING RULES — two drug entries refer to the same medication if:\n"
            "  • The names are identical or differ only by OCR noise / "
            "capitalisation / accents.\n"
            "  • One name is the brand name and the other is the INN (generic) "
            "for the same active substance "
            "(e.g., DOLIPRANE = Paracétamol, AMOXIL = Amoxicilline).\n"
            "  • One name includes the dosage strength inline and the other "
            "separates it (e.g., 'Metformine 850mg' vs 'Metformine' with "
            "dosage='850mg').\n"
            "  • Minor OCR artefacts: a digit confused for a letter, a missing "
            "accent, a hyphen vs space.\n\n"

            "DO NOT penalise the model for finding EXTRA medications beyond "
            "the ground truth — this metric measures recall only.\n\n"

            "SCORING:\n"
            "  Score = (number of ground-truth drugs found) / "
            "(total ground-truth drugs)\n\n"
            "  1.0 — Every ground-truth drug was found.\n"
            "  0.5 — Half of the drugs were found.\n"
            "  0.0 — No drugs from the ground truth were found.\n\n"

            "If the ground truth contains zero medications, score 1.0.\n"
            "Provide a brief explanation listing which drugs were found and "
            "which were missed."
        ),
    )


# ─────────────────────────────────────────────────────────────
# Convenience: instantiate all four metrics at once
# ─────────────────────────────────────────────────────────────
def get_all_metrics():
    """
    Returns all four evaluation metrics as a list.
    Use with deepeval.evaluate() or assert_test().

    Metrics 1 & 2 are deterministic (no API calls).
    Metrics 3 & 4 call an LLM judge via GEval (requires OPENAI_API_KEY or
    whichever judge model you configure in build_*_metric()).
    """
    return [
        JSONValidityMetric(),
        SchemaComplianceMetric(),
        build_entity_accuracy_metric(),
        build_medication_recall_metric(),
    ]

## Running the evaluation

In [ ]:
CURRENT_TIME = "t" + str(time.localtime().tm_mday) + "-" + str(time.localtime().tm_mon) + "-" + str(time.localtime().tm_year) + "_"+ str(time.localtime().tm_hour) + "-" + str(time.localtime().tm_min)

In [ ]:
def check_models_available(models: list) -> dict:
    try:
        # Get the list of models currently on the system
        response = ollama.list()
        print(response)
        available = {m.model for m in response.models}
        
    except Exception as e:
        print(f"⚠️  Could not connect to Ollama or parse list: {e}")
        return {m: False for m in models}

    status = {}
    available_models = []
    for m in models:
        # Check for exact match or if the requested string is part of the tag
        found = m in available or any(m in a for a in available)
        if found: available_models.append(m)
        status[m] = found
        icon = "✅" if found else "❌"
        print(f"  {icon} {m}")
        
    return available_models

def call_ollama(model: str, ocr_text: str, prompt: str, save_output: bool) -> dict:
    
    # loading the prompt template and filling in the OCR text

    user_msg = USER_TEMPLATE.format(ocr_text=ocr_text)
    system_prompt = SYSTEM_PROMPT.format(prompt=prompt)
    t0 = time.time()
    metadata = {
        "model": model,
        "raw_output": "",
        "latency_sec": 0.0,
        "error": None
    }
    parsed_data = None
    
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_msg},
            ],
            format="json", # forces the model to respond in a JSON format
            options={"temperature": 0.0},
        )
        
        metadata["latency_sec"] = time.time() - t0
        raw = response["message"]["content"].strip()
        metadata["raw_output"] = raw

        # Strip markdown fences
        clean = re.sub(r"^```[a-z]*\n?", "", raw)
        clean = re.sub(r"\n?```$", "", clean).strip()

        parsed_data = json.loads(clean)

    except json.JSONDecodeError as e:
        metadata["error"] = f"JSON parse error: {e}"
        metadata["latency_sec"] = time.time() - t0
    except Exception as e:
        metadata["error"] = str(e)
        metadata["latency_sec"] = time.time() - t0

    if save_output:
        # Sanitize model name for filenames (remove special characters)
        safe_name = re.sub(r'[^a-zA-Z0-9]', '_', model)
        
        result_file = f"{safe_name}_result.json"
        metadata_file = f"{safe_name}_metadata.json"
        results_folder_name = f'{CURRENT_TIME}'
        %mkdir -p {results_folder_name}

        %cd {results_folder_name}
    
        with open(result_file, "w", encoding="utf-8") as f:
            json.dump(parsed_data, f, indent=4)
    
        with open(metadata_file, "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=4)
        %cd ..

    return {
        "parsed_json": parsed_data,
        "raw_output": metadata["raw_output"],
        "latency_sec": metadata["latency_sec"],
        "error": metadata["error"],
        "success": metadata["error"] is None
    }

In [ ]:
import asyncio
from deepeval.test_case import LLMTestCase
from deepeval.metrics import GEval
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TimeElapsedColumn
from rich.console import Console

console = Console()

# ── Instantiate all 4 metrics ─────────────────────────────────
# Metrics 1 & 2: deterministic, no API calls
# Metrics 3 & 4: GEval — LLM judge via OpenAI (needs OPENAI_API_KEY)
METRICS = get_all_metrics()

# ── Helper: score one test_case across all metrics ────────────
async def score_test_case(test_case: LLMTestCase) -> dict:
    """
    Runs all metrics against a single test case.
    Deterministic metrics (1 & 2) are called synchronously.
    GEval metrics (3 & 4) are awaited concurrently for speed.
    Returns {metric_name: score} dict.
    """
    scores = {}
    sync_metrics  = [m for m in METRICS if not isinstance(m, GEval)]
    async_metrics = [m for m in METRICS if isinstance(m, GEval)]

    # Score deterministic metrics immediately
    for metric in sync_metrics:
        metric.measure(test_case)
        scores[metric.name] = round(metric.score, 4)

    # Score GEval metrics concurrently (each calls the judge LLM once)
    if async_metrics:
        tasks = [m.a_measure(test_case) for m in async_metrics]
        results_async = await asyncio.gather(*tasks, return_exceptions=True)
        for metric, result in zip(async_metrics, results_async):
            if isinstance(result, Exception):
                console.print(f"[red]⚠ GEval error for {metric.name}: {result}[/]")
                scores[metric.name] = 0.0
            else:
                scores[metric.name] = round(metric.score, 4)

    return scores


# ── Main evaluation loop ──────────────────────────────────────
results = {}  # results[model][example_id] = {scores, latency, error, ...}

AVAILABLE_MODELS = check_models_available(MODELS_TO_EVAL)
print(AVAILABLE_MODELS)

total_calls = len(AVAILABLE_MODELS) * len(EXAMPLES)
console.print(f"\n[bold cyan]Starting evaluation: {len(AVAILABLE_MODELS)} models × {len(EXAMPLES)} examples = {total_calls} inference calls[/]\n")
console.print("[dim]Metrics 1 & 2 are deterministic. Metrics 3 & 4 call an LLM judge — expect ~2–5s per example.[/]\n")

async def run_evaluation():
    with Progress(
        SpinnerColumn(),
        TextColumn("[bold blue]{task.description}"),
        BarColumn(),
        TextColumn("{task.completed}/{task.total}"),
        TimeElapsedColumn(),
        console=console,
    ) as progress:
        task = progress.add_task("Evaluating...", total=total_calls)

        for model in AVAILABLE_MODELS:
            results[model] = {}
            for ex in EXAMPLES:
                progress.update(task, description=f"{model} | {ex['data']['id']}")

                # 1. Inference (Ollama)
                infer = call_ollama(model, ex["data"]["ocr_text"], prompt, save_output=True)

                # 2. Build DeepEval test case
                actual = (
                    infer["raw_output"]
                    if infer["parsed_json"] is None
                    else json.dumps(infer["parsed_json"], ensure_ascii=False)
                )
                test_case = LLMTestCase(
                    input=USER_TEMPLATE.format(ocr_text=ex["data"]["ocr_text"]),
                    actual_output=actual,
                    expected_output=json.dumps(ex["data"]["ground_truth"], ensure_ascii=False),
                )

                # 3. Score all metrics (sync + async GEval)
                metric_scores = await score_test_case(test_case)

                results[model][ex["data"]["id"]] = {
                    "scores":  metric_scores,
                    "latency": round(infer["latency_sec"], 2),
                    "error":   infer["error"],
                    "parsed":  infer["parsed_json"],
                    "raw":     infer["raw_output"][:300],
                }
                progress.advance(task)

# Colab / Jupyter already runs an event loop — use await directly
await run_evaluation()

console.print("\n[bold green]✅ Evaluation complete![/]")


In [ ]:
import pandas as pd

METRIC_NAMES = [m.name for m in METRICS]  # auto-picks up new GEval names

rows = []
for model in AVAILABLE_MODELS:
    model_results = results[model]
    n = len(model_results)
    avg_scores = {mn: 0.0 for mn in METRIC_NAMES}
    avg_latency = 0.0
    errors = 0

    for ex_id, res in model_results.items():
        for mn in METRIC_NAMES:
            avg_scores[mn] += res["scores"].get(mn, 0.0)
        avg_latency += res["latency"]
        if res["error"]:
            errors += 1

    row = {"Model": model}
    for mn in METRIC_NAMES:
        row[mn] = round(avg_scores[mn] / n, 3)
    row["Avg Latency (s)"] = round(avg_latency / n, 2)
    row["Overall"] = round(sum(row[mn] for mn in METRIC_NAMES) / len(METRIC_NAMES), 3)
    rows.append(row)

df = pd.DataFrame(rows).sort_values("Overall", ascending=False).reset_index(drop=True)
df.index += 1
df.index.name = "Rank"

def highlight(val, threshold=0.7):
    if isinstance(val, float):
        if val >= 0.9:   return "background-color: #2d6a4f; color: white"
        elif val >= 0.7: return "background-color: #52b788"
        elif val >= 0.5: return "background-color: #f4a261"
        else:            return "background-color: #e63946; color: white"
    return ""

styled = df.style.map(highlight, subset=METRIC_NAMES + ["Overall"])  # .map() replaces deprecated .applymap()
styled


In [ ]:
# Pivot: for a chosen metric, show all models × all examples
# Available metric names: ['JSON Validity', 'Schema Compliance', 'Semantic Entity Accuracy', 'Semantic Medication Recall']
TARGET_METRIC = "Semantic Entity Accuracy"  # ← change to any metric name above

pivot_rows = []
for model in AVAILABLE_MODELS:
    row = {"Model": model}
    for ex in EXAMPLES:
        score = results[model][ex["data"]["id"]]["scores"].get(TARGET_METRIC, None)
        row[ex["data"]["id"]] = round(score, 3) if score is not None else "ERR"
    pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).set_index("Model")
print(f"\n📊 {TARGET_METRIC} — per-example scores:")
pivot_df.style.map(highlight, subset=list(pivot_df.columns))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Resolve the 'quality' column: prefer Semantic Entity Accuracy ──
QUALITY_METRIC = "Semantic Entity Accuracy" if "Semantic Entity Accuracy" in df.columns else METRIC_NAMES[2]

# ── Plot 1: Overall score bar chart ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.RdYlGn(df["Overall"].values)
bars = axes[0].barh(df["Model"], df["Overall"], color=colors, edgecolor="white", height=0.6)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Score", fontsize=11)
axes[0].set_title("Overall Score (avg of all metrics)", fontsize=13, fontweight="bold")
axes[0].axvline(0.7, color="orange", linestyle="--", linewidth=1, label="threshold 0.7")
axes[0].legend()
for bar, val in zip(bars, df["Overall"]):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.2f}", va="center", fontsize=9)

# ── Plot 2: Latency vs Semantic Entity Accuracy scatter ────────
axes[1].scatter(df["Avg Latency (s)"], df[QUALITY_METRIC],
                c=df["Overall"], cmap="RdYlGn", s=120, edgecolors="black", linewidths=0.8)
for _, row_ in df.iterrows():
    axes[1].annotate(row_["Model"].split(":")[0],
                     (row_["Avg Latency (s)"], row_[QUALITY_METRIC]),
                     textcoords="offset points", xytext=(5, 5), fontsize=8)
axes[1].set_xlabel("Avg Latency (s)", fontsize=11)
axes[1].set_ylabel(QUALITY_METRIC, fontsize=11)
axes[1].set_title("Quality vs Speed Trade-off", fontsize=13, fontweight="bold")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("/tmp/eval_plots1.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Plot 3: Radar / spider chart per model ─────────────────────
from matplotlib.patches import FancyArrowPatch

RADAR_METRICS = METRIC_NAMES  # all 4 metrics
N = len(RADAR_METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
cmap = plt.cm.tab10

for i, model in enumerate(AVAILABLE_MODELS):
    vals = [df[df["Model"] == model][mn].values[0] for mn in RADAR_METRICS]
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=1.8, color=cmap(i), label=model)
    ax.fill(angles, vals, alpha=0.08, color=cmap(i))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_METRICS, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_title("Model Capability Radar", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/eval_plots2.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Summary CSV ────────────────────────────────────────────────
df.to_csv("ocr_eval_summary.csv")
print("Saved: ocr_eval_summary.csv")

# ── Detailed CSV (every model × example × metric) ─────────────
detail_rows = []
for model in AVAILABLE_MODELS:
    for ex in EXAMPLES:
        r = results[model][ex["data"]["id"]]
        row = {"model": model, "example_id": ex["data"]["id"],
               "latency_s": r["latency"]}
        row.update(r["scores"])
        detail_rows.append(row)

detail_df = pd.DataFrame(detail_rows)
detail_df.to_csv("ocr_eval_detailed.csv", index=False)
print("Saved: ocr_eval_detailed.csv")

# ── Print final ranking ────────────────────────────────────────
QUALITY_METRIC = "Semantic Entity Accuracy" if "Semantic Entity Accuracy" in df.columns else METRIC_NAMES[2]

console.print("\n[bold cyan]🏆 Final Ranking[/]")
medals = ["🥇", "🥈", "🥉"]
for rank, (_, row_) in enumerate(df.iterrows(), 1):
    medal = medals[rank - 1] if rank <= 3 else "  "
    console.print(
        f"{medal} #{rank:2d}  [bold]{row_['Model']:<35}[/]  "
        f"Overall: {row_['Overall']:.3f}  "
        f"{QUALITY_METRIC}: {row_[QUALITY_METRIC]:.3f}  "
        f"Latency: {row_['Avg Latency (s)']}s"
    )


In [ ]:
results_folder = pathlib.Path(CURRENT_TIME)
results_folder.mkdir(parents=True, exist_ok=True)

# Save the raw evaluation results dict
with open(results_folder / "evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

# Save summary dataframe if it exists, otherwise rebuild it
summary_csv = results_folder / "ocr_eval_summary.csv"
if "df" in globals():
    df.to_csv(summary_csv)
else:
    rows = []
    for model in AVAILABLE_MODELS:
        model_results = results[model]
        n = len(model_results)
        avg_scores = {mn: 0.0 for mn in METRIC_NAMES}
        avg_latency = 0.0

        for ex_id, res in model_results.items():
            for mn in METRIC_NAMES:
                avg_scores[mn] += res["scores"].get(mn, 0.0)
            avg_latency += res["latency"]

        row = {"Model": model}
        for mn in METRIC_NAMES:
            row[mn] = round(avg_scores[mn] / n, 3)
        row["Avg Latency (s)"] = round(avg_latency / n, 2)
        row["Overall"] = round(sum(row[mn] for mn in METRIC_NAMES) / len(METRIC_NAMES), 3)
        rows.append(row)

    df = pd.DataFrame(rows).sort_values("Overall", ascending=False).reset_index(drop=True)
    df.index += 1
    df.index.name = "Rank"
    df.to_csv(summary_csv)

# Save detailed per-model/per-example scores
detail_rows = []
for model in AVAILABLE_MODELS:
    for ex in EXAMPLES:
        r = results[model][ex["data"]["id"]]
        row = {"model": model, "example_id": ex["data"]["id"], "latency_s": r["latency"]}
        row.update(r["scores"])
        detail_rows.append(row)

detail_df = pd.DataFrame(detail_rows)
detail_df.to_csv(results_folder / "ocr_eval_detailed.csv", index=False)

print(f"Saved evaluation results to: {results_folder}")